# 🚲 Bicycle RTI Hackathon — One-Click Deployment

This notebook deploys the **complete** Bicycle Real-Time Intelligence solution into your workspace.

## What Gets Deployed (26 items)
| Stage | Items |
|-------|-------|
| 1. Infrastructure | 4 Lakehouses, 1 Eventhouse, 1 KQL Database |
| 2. Compute & Ingestion | 9 Notebooks, 2 Eventstreams |
| 3. Analytics | 2 Semantic Models, 1 Pipeline |
| 4. Presentation | 1 Report, 1 KQL Dashboard, 2 Data Agents, 2 Activators |

## Prerequisites
- A Fabric workspace with **F64 or higher** capacity
- **Contributor** or higher permissions on the workspace
- If the GitHub repo is **private**: a GitHub Personal Access Token (PAT) with `repo` scope

## Instructions
1. Run **Cell 1** to install dependencies (auto-restarts Python)
2. Run **Cell 2** to configure and deploy all 26 items
3. Run **Cell 3** to validate deployment
4. See **Cell 4** for post-deployment manual steps

In [ ]:
# =============================================================
# CELL 1 — Install dependencies
# =============================================================
# Pip dependency warnings from nni/mlflow/datasets are harmless —
# they are pre-installed Fabric runtime packages, not ours.

%pip install fabric-cicd --quiet

# Restart Python so the newly installed package is importable
try:
    import notebookutils
    notebookutils.session.restartPython()
except NameError:
    print("⚠️  notebookutils not available — restart the kernel manually before Cell 2.")

In [ ]:
# =============================================================
# CELL 2 — Download repo + Deploy all 26 items
# =============================================================
import os, zipfile, shutil, tempfile, requests
import sempy.fabric as fabric
from fabric_cicd import FabricWorkspace, publish_all_items

# ======== CONFIGURATION — edit these if you forked ========
REPO_OWNER = "kwamesefah_microsoft"
REPO_NAME  = "RTI-Hackathon-Demo"
BRANCH     = "main"

# For PRIVATE repos: paste a GitHub PAT with 'repo' scope.
# For PUBLIC repos: leave as empty string "".
GITHUB_TOKEN = ""  # <-- set your PAT here if repo is private
# ==========================================================

# --- Step 1: Download the repo zip from GitHub ---
print("📥 Step 1: Downloading repo from GitHub...")
zip_url = f"https://api.github.com/repos/{REPO_OWNER}/{REPO_NAME}/zipball/{BRANCH}"
headers = {"Accept": "application/vnd.github+json"}
if GITHUB_TOKEN:
    headers["Authorization"] = f"Bearer {GITHUB_TOKEN}"

resp = requests.get(zip_url, headers=headers, stream=True)
resp.raise_for_status()

work_dir = tempfile.mkdtemp(prefix="rti_deploy_")
zip_path = os.path.join(work_dir, "repo.zip")
with open(zip_path, "wb") as f:
    for chunk in resp.iter_content(chunk_size=8192):
        f.write(chunk)
print(f"   Downloaded {os.path.getsize(zip_path):,} bytes")

# --- Step 2: Extract and locate workspace/ folder ---
print("📦 Step 2: Extracting...")
with zipfile.ZipFile(zip_path, "r") as zf:
    zf.extractall(work_dir)

# GitHub zips nest under a random folder name like "owner-repo-sha/"
extracted_dirs = [d for d in os.listdir(work_dir) if os.path.isdir(os.path.join(work_dir, d))]
repo_root = os.path.join(work_dir, extracted_dirs[0])
workspace_dir = os.path.join(repo_root, "workspace")

item_count = len([d for d in os.listdir(workspace_dir) if os.path.isdir(os.path.join(workspace_dir, d))])
print(f"   Found {item_count} items in workspace/")

# --- Step 3: Deploy with fabric-cicd (staged) ---
ws_id = fabric.get_workspace_id()
print(f"\n🚀 Step 3: Deploying to workspace {ws_id}")

# Stage 1: Infrastructure
print("\n--- Stage 1/4: Infrastructure (Lakehouses, Eventhouse, KQL DB) ---")
ws1 = FabricWorkspace(
    workspace_id=ws_id,
    repository_directory=workspace_dir,
    item_type_in_scope=["Lakehouse", "Eventhouse", "KQLDatabase"],
)
publish_all_items(ws1)
print("   ✅ Stage 1 complete")

# Stage 2: Compute + Ingestion
print("\n--- Stage 2/4: Compute & Ingestion (Notebooks, Eventstreams) ---")
ws2 = FabricWorkspace(
    workspace_id=ws_id,
    repository_directory=workspace_dir,
    item_type_in_scope=["Notebook", "Eventstream"],
)
publish_all_items(ws2)
print("   ✅ Stage 2 complete")

# Stage 3: Analytics
print("\n--- Stage 3/4: Analytics (Semantic Models, Pipeline) ---")
ws3 = FabricWorkspace(
    workspace_id=ws_id,
    repository_directory=workspace_dir,
    item_type_in_scope=["SemanticModel", "DataPipeline"],
)
publish_all_items(ws3)
print("   ✅ Stage 3 complete")

# Stage 4: Presentation + AI
print("\n--- Stage 4/4: Presentation & AI (Report, Dashboard, Agents, Activators) ---")
ws4 = FabricWorkspace(
    workspace_id=ws_id,
    repository_directory=workspace_dir,
    item_type_in_scope=["Report", "KQLDashboard", "DataAgent", "Reflex"],
)
publish_all_items(ws4)
print("   ✅ Stage 4 complete")

# --- Cleanup ---
shutil.rmtree(work_dir, ignore_errors=True)

print("\n" + "=" * 60)
print("✅ DEPLOYMENT COMPLETE — 26 items deployed")
print("=" * 60)
print("\nNext: Run Cell 3 to validate, then see Cell 4 for manual steps.")

In [ ]:
# =============================================================
# CELL 3 — Validate Deployment
# =============================================================
import sempy.fabric as fabric

ws_id = fabric.get_workspace_id()
print(f"Workspace: {ws_id}")

# List all items
items = fabric.list_items()
print(f"\nTotal items: {len(items)}")

# Group by type
from collections import Counter
type_counts = Counter(row['Type'] for _, row in items.iterrows())
print("\nItems by type:")
for t, c in sorted(type_counts.items()):
    print(f"  {t}: {c}")

# Check critical items
expected = [
    "bicycles_gold", "bikerental_bronze_raw", "bicycles_silver", "weather_bronze_raw",
    "bikerentaleventhouse",
    "RTIbikeRental", "RTI-WeatherDemo",
    "PL_BicycleRTI_Medallion",
    "Bicycle RTI Analytics", "Bicycle Ontology Model",
    "Bicycle Fleet Intelligence Agent", "ontology data agent",
]
names = set(items['Display Name'])
print("\nCritical item check:")
for e in expected:
    status = "✅" if e in names else "❌ MISSING"
    print(f"  {status} {e}")

## Post-Deployment Manual Steps

The automated deployment handles 26 items. The following items require
manual setup because their item types are not yet supported by fabric-cicd:

### 1. Run the Pipeline (First Load)
1. Go to **PL_BicycleRTI_Medallion** pipeline
2. Click **Run** — this processes: Silver → Gold → ML → Ontology → SM Refresh
3. Wait ~15-25 min for completion

### 2. Ontology + Graph Model (Post-Deploy Notebook)
1. Upload and run `Post_Deploy_Setup.ipynb` to create:
   - Bicycle_Ontology_Model_New (Ontology)
   - Bicycle_Ontology_Model_New_graph (GraphModel)
   - Cycling-Campaign-Agent (OperationsAgent)

### 3. Verify Eventstreams
Both eventstreams should auto-start after deployment:
- **RTIbikeRental** — Bicycle sample data → Lakehouse + Eventhouse
- **RTI-WeatherDemo** — Real-time weather → Eventhouse

If not running, open each Eventstream and click **Start**.

### 4. Activator Rules
Both Reflex/Activator items are deployed as shells. Add alert rules:
- **BicycleFleet_Activator**: Low stock alert, high demand surge
- **Cycling Campaign Activator**: Campaign trigger based on weather

See `docs/ACTIVATOR_SETUP.md` in the repo for rule definitions.